# Lab Final : เทรน YOLO แบบ Polygon (เส้นรอบรูปวัตถุ)
**โครงเดียวกับ `lab_2.ipynb`** · โมเดล `yolo26n-seg.pt` · task = **segment (Instance Segmentation)**

### notebook นี้ทำอะไร (ไล่จากบนลงล่าง)
| ขั้น | ทำอะไร | ได้อะไร |
|---|---|---|
| 0 | import + ตั้งค่า + เช็คเครื่อง | พร้อมใช้งาน |
| 1 | ดู dataset (`poly.yaml`, จำนวนภาพ/object, หน้าตาไฟล์ label) | รู้ว่าข้อมูลถูกต้อง |
| 2 | โหลดโมเดล pretrained แล้วลองทายก่อนเทรน | เห็นว่าก่อนเทรนโมเดลรู้จัก 80 คลาสของ COCO |
| 3 | **เทรน** 20 epoch | `runs/poly/weights/best.pt` |
| 4 | ดูผลการเทรน + วัดผลชุด test | กราฟ, confusion matrix, mAP |
| 5 | Predict ด้วย weight ใหม่ | `runs/predict_poly/` (ผลการ Predict ส่งอาจารย์) |
| 6 | **Export โมเดลไว้ใช้กับเว็บ** | `weights/poly_best.pt` (หลังบ้าน) + `web/frontend/models/poly.onnx` (หน้าเว็บ) |

**คลาส:** 0 airplane · 1 boat · 2 car · 3 motorcycle

> ⚠️ ต้องเปิด notebook นี้จากโฟลเดอร์ `final_dip/` เพราะ path ทุกตัวอ้างอิงจากโฟลเดอร์นี้

---
## ขั้น 0 : เตรียมเครื่องมือ

**อธิบาย Cell 1 :** import ไลบรารีที่ใช้ทั้งหมด เหมือนใน lab
- `ultralytics` ไลบรารีของ YOLO ใช้ทั้งโหลดโมเดล เทรน และทำนาย
- `cv2` (OpenCV) อ่านไฟล์ภาพ และวาดกรอบหรือเส้นลงบนภาพ
- `matplotlib.pyplot` แสดงภาพใน notebook
- `os`, `glob`, `shutil` จัดการไฟล์และโฟลเดอร์ (หาไฟล์ คัดลอก ย้าย)

In [ ]:
# ===== Cell 1 : import ไลบรารี (เหมือน lab) =====
from IPython import display      # ใช้ล้างหน้าจอ output
import ultralytics               # ไลบรารี YOLO
import cv2                       # OpenCV อ่าน/วาดภาพ
import matplotlib.pyplot as plt  # แสดงภาพใน notebook
import os, glob, shutil          # จัดการไฟล์

**อธิบาย Cell 2 :**
- `KMP_DUPLICATE_LIB_OK = TRUE` แก้ error "OMP: Error #15" ที่มักเกิดบน Windows เมื่อมีไลบรารีคำนวณหลายตัวโหลดซ้ำกัน (ใน lab ก็ใส่บรรทัดนี้)
- `RUNS` คือโฟลเดอร์เก็บผลลัพธ์ ต้องใช้ **path เต็ม** เพราะถ้าเขียนแค่ `"runs"` ultralytics จะไปสร้างไว้ลึกที่ `runs/detect/runs/...` ทำให้หาไฟล์ยาก

In [ ]:
# ===== Cell 2 : ตั้งค่า =====
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # แก้ error OMP บน Windows (เหมือน lab)

RUNS = os.path.abspath("runs")                # โฟลเดอร์เก็บผล (path เต็ม)
os.makedirs("weights", exist_ok=True)         # โฟลเดอร์เก็บ weight ไว้ส่งงาน/ให้เว็บใช้
print("ผลลัพธ์จะเก็บที่:", RUNS)

**อธิบาย Cell 3 :** `ultralytics.checks()` แสดงเวอร์ชันของ ultralytics, Python, PyTorch และบอกว่าเครื่องนี้ใช้ CPU หรือ GPU (เครื่องนี้ใช้ CPU จึงเทรนช้ากว่า GPU)

In [ ]:
# ===== Cell 3 : เช็คเวอร์ชัน / CPU / GPU =====
display.clear_output()
ultralytics.checks()

---
## ขั้น 1 : ดู dataset ก่อนเทรน

**อธิบาย Cell 4 :** ไฟล์ `poly.yaml` บอก YOLO ว่า
- `path` dataset อยู่โฟลเดอร์ไหน
- `train` / `val` / `test` โฟลเดอร์ย่อยของแต่ละชุด (ข้างในมี `images/` กับ `labels/`)
- `names` ชื่อคลาสตามลำดับเลข 0–3 (**ต้องใช้คำว่า `names` ที่มี s** ใน lab เดิมพิมพ์ว่า `name` ทำให้ YOLO ไม่รู้ชื่อคลาส)

แบ่งข้อมูล 3 ชุด
- **train (70%)** ใช้สอนโมเดล
- **val (20%)** ใช้วัดผลระหว่างเทรนทุก epoch เพื่อเลือก weight ที่ดีที่สุด
- **test (10%)** เก็บไว้วัดผลตอนจบ โมเดลไม่เคยเห็นเลย

ตารางจะนับจำนวน object ต่อคลาสในชุด train ด้วย โจทย์กำหนดว่าต้องมีอย่างน้อย 100

In [ ]:
# ===== Cell 4 : ดูไฟล์ yaml + นับจำนวนภาพและ object =====
print(open("poly.yaml", encoding="utf-8").read())

names = ["airplane", "boat", "car", "motorcycle"]
for split in ["train", "val", "test"]:
    labels = glob.glob(f"datasets/yolo_poly/{split}/labels/*.txt")
    count = [0, 0, 0, 0]
    for f in labels:                                   # เปิดทุกไฟล์ label
        for line in open(f).read().splitlines():       # 1 บรรทัด = 1 object
            count[int(line.split()[0])] += 1           # ตัวเลขแรกของบรรทัด = เลขคลาส
    detail = ", ".join(f"{n}={c}" for n, c in zip(names, count))
    print(f"{split:<5} : {len(labels):>3} ภาพ | object: {detail}")

**อธิบาย Cell 5 :** ดูหน้าตาไฟล์ label แบบ Polygon แล้ววาดกลับลงบนภาพ เพื่อพิสูจน์ว่า label ถูกตำแหน่ง

1 บรรทัด = 1 object รูปแบบ `class  x1 y1  x2 y2  x3 y3 ...` คือพิกัดทุกจุดรอบวัตถุเรียงต่อกัน
- ทุกค่า normalize แล้ว (0–1) ต้องคูณกลับด้วยความกว้าง/สูงของภาพก่อนวาด
- polygon ละเอียดกว่ากรอบสี่เหลี่ยม เพราะบอก "รูปร่าง" ของวัตถุได้ด้วย

In [ ]:
# ===== Cell 5 : ดูไฟล์ label 1 ไฟล์ แล้ววาดกลับลงภาพ =====
import numpy as np
img_path = sorted(glob.glob("datasets/yolo_poly/train/images/car_*.jpg"))[0]
lbl_path = img_path.replace("images", "labels").replace(".jpg", ".txt")
print("ไฟล์:", lbl_path)
print(open(lbl_path).read()[:300], "...")

im = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
h, w = im.shape[:2]
for line in open(lbl_path).read().splitlines():
    v = list(map(float, line.split()))
    pts = (np.array(v[1:]).reshape(-1, 2) * [w, h]).astype(np.int32)   # คูณกลับเป็น pixel
    cv2.polylines(im, [pts.reshape(-1, 1, 2)], True, (255, 0, 255), 3)
    print(f"object คลาส {int(v[0])} มี {len(pts)} จุด")
plt.figure(figsize=(9, 6)); plt.imshow(im); plt.title("label from txt"); plt.axis("off"); plt.show()

---
## ขั้น 2 : โหลดโมเดล pretrained แล้วลองทายก่อนเทรน

**อธิบาย Cell 6 :** `ultralytics.YOLO('yolo26n-seg.yaml').load('yolo26n-seg.pt')` (แบบเดียวกับ lab)
- `yolo26n-seg.yaml` = **โครงสร้าง** ของโมเดล (มีกี่ชั้น ต่อกันยังไง)
- `.load('yolo26n-seg.pt')` = ใส่ **ค่า weight** ที่ถูกฝึกมาแล้วบน dataset COCO (80 คลาส เช่น คน รถ หมา แมว)
- ตัว `n` (nano) คือรุ่นเล็กที่สุด เร็วและเหมาะกับ CPU
- `-seg` คือรุ่น segmentation ทายได้ทั้งกรอบและรูปร่าง (mask) ของวัตถุ

In [ ]:
# ===== Cell 6 : โหลดโมเดล =====
model = ultralytics.YOLO('yolo26n-seg.yaml').load('yolo26n-seg.pt')
print("จำนวนคลาสที่โมเดลรู้จักตอนนี้:", len(model.names))

**อธิบาย Cell 7 :** ลองให้โมเดลที่ยังไม่ได้เทรนทายภาพในชุด test (โค้ดวาดผลเหมือน lab)
- `model.predict(source=im_src)` ส่งภาพเข้าโมเดล ได้ผลลัพธ์เป็น list (1 ภาพ = 1 ตัว) จึงใช้ `result[0]`
- ภาพจาก `cv2.imread` เป็นสีแบบ **BGR** ต้องแปลงเป็น **RGB** ก่อนแสดงด้วย matplotlib ไม่งั้นสีจะเพี้ยน
- `result[0].masks.xy` = จุด polygon ของแต่ละวัตถุ ใช้ `cv2.polylines` ลากเส้นเชื่อมจุด
- `result[0].boxes` = กรอบ, คลาส และความมั่นใจของวัตถุเดียวกัน (ลำดับตรงกับ masks)

ตอนนี้โมเดลยังใช้ชื่อคลาสของ COCO (80 คลาส) หลังเทรนจะเหลือแค่ 4 คลาสของเรา

In [ ]:
# ===== Cell 7 : ลอง Predict ก่อนเทรน =====
path = sorted(glob.glob('datasets/yolo_poly/test/images/*.jpg'))[0]
im_src = cv2.imread(path)                              # อ่านภาพ (BGR)
result = model.predict(source=im_src)                  # ให้โมเดลทาย
im_result = cv2.cvtColor(im_src, cv2.COLOR_BGR2RGB)    # แปลงสีเป็น RGB ไว้แสดงผล

if result[0].masks is not None:                       # ถ้าไม่เจอวัตถุเลย masks จะเป็น None
    for i in range(len(result[0].masks.xy)):          # วนทีละวัตถุ
        class_id = int(result[0].boxes.cls[i].item()) # เลขคลาส
        conf = round(result[0].boxes.conf[i].item()*100, 2)   # ความมั่นใจ %
        x = result[0].masks.xy[i].astype(int)         # จุดของ polygon [[x, y], ...] หน่วย pixel
        pts = x.reshape((-1, 1, 2))                   # จัดรูปแบบให้ตรงกับที่ cv2.polylines ต้องการ
        im_result = cv2.polylines(im_result, [pts], isClosed=True, color=(255, 0, 255), thickness=3)
        im_result = cv2.putText(im_result, f"{result[0].names[class_id]} {conf}%", (int(x[0][0]), max(int(x[0][1]) - 8, 20)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2, cv2.LINE_AA)

plt.figure(figsize=(10, 7))
plt.imshow(im_result)
plt.title("before training (COCO 80 classes)")
plt.show()

---
## ขั้น 3 : เทรน (Transfer Learning)

**แนวคิด:** ไม่เริ่มจากศูนย์ แต่เอาโมเดลที่เก่งอยู่แล้ว (ฝึกบน COCO) มา **ฝึกต่อ** ให้รู้จักแค่ 4 คลาสของเรา จึงใช้ภาพน้อย (คลาสละ 150 ภาพ) และเทรนเร็ว

**อธิบาย Cell 8 :** แต่ละพารามิเตอร์ของ `model.train(...)`
| พารามิเตอร์ | ค่า | ความหมาย |
|---|---|---|
| `data` | `"poly.yaml"` | ไฟล์บอกว่า dataset อยู่ไหน มีคลาสอะไร |
| `epochs` | 20 | ดูข้อมูลชุด train ครบ 20 รอบ |
| `imgsz` | 512 | ย่อภาพเป็น 512 pixel ก่อนเข้าโมเดล (เล็กลง = เร็วขึ้น) |
| `batch` | 16 | ป้อนภาพทีละ 16 ภาพ แล้วปรับ weight 1 ครั้ง |
| `patience` | 15 | ถ้า 15 epoch ติดกันไม่ดีขึ้น ให้หยุดก่อนกำหนด (early stopping) |
| `workers` | 2 | จำนวน process ที่ช่วยโหลดภาพ |
| `device` | `"cpu"` | เทรนด้วย CPU (ถ้ามี GPU ใส่ `0`) |
| `seed` | 42 | ตั้งค่าการสุ่มให้เหมือนเดิม รันกี่ครั้งก็ได้ผลใกล้เคียงกัน |
| `project`, `name` | `RUNS`, `"poly"` | เก็บผลที่ `runs/poly/` |

**ระหว่างเทรน** แต่ละ epoch จะแสดงค่า loss (ความผิดพลาด ยิ่งน้อยยิ่งดี)
- `box_loss` = กรอบผิดตำแหน่งแค่ไหน
- `seg_loss` = รูปร่าง (mask) ผิดแค่ไหน
- `cls_loss` = ทายคลาสผิดแค่ไหน

จบแต่ละ epoch จะวัดผลกับชุด val (P, R, mAP50) แล้วเก็บ weight ที่ดีที่สุดไว้เป็น `best.pt` ตอนจบ cell นี้จะ copy `best.pt` ไปไว้ที่ `weights/poly_best.pt`

⏱ บน CPU ใช้เวลาประมาณ 50 นาที

In [ ]:
# ===== Cell 8 : เทรน =====
model = ultralytics.YOLO("yolo26n-seg.pt")                  # เริ่มจาก weight pretrained
model.train(data="poly.yaml", epochs=20, imgsz=512, batch=16, patience=15, workers=2,
            device="cpu", seed=42, project=RUNS, name="poly", exist_ok=True)

# copy weight ที่ดีที่สุดไปไว้ที่ weights/ (ส่งอาจารย์ + ให้เว็บใช้)
shutil.copy("runs/poly/weights/best.pt", "weights/poly_best.pt")
print("saved -> weights/poly_best.pt")

---
## ขั้น 4 : ดูผลการเทรน

**อธิบาย Cell 9 : `results.png`** กราฟที่ ultralytics วาดให้อัตโนมัติ แกน x = epoch
- กราฟ `train/...loss` และ `val/...loss` ควร **ลดลง** แปลว่าโมเดลเรียนรู้ได้ (ถ้า train ลดแต่ val เพิ่ม แปลว่าจำข้อมูลมากเกินไป หรือ overfitting)
- กราฟ `metrics/precision`, `recall` และ `mAP50` ควร **เพิ่มขึ้น**
- กราฟที่ลงท้ายด้วย (B) คือผลของกรอบ (Box) และ (M) คือผลของ Mask (รูปร่าง)

In [ ]:
# ===== Cell 9 : กราฟผลการเทรน =====
img = cv2.cvtColor(cv2.imread("runs/poly/results.png"), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(18, 9)); plt.imshow(img); plt.axis("off"); plt.title("runs/poly/results.png"); plt.show()

**อธิบาย Cell 10 : Confusion Matrix** (แบบ normalize เป็นสัดส่วน 0–1)
- แกนตั้ง = คลาสที่โมเดล **ทาย**, แกนนอน = คลาส **จริง**
- **เส้นทแยงมุม** = ทายถูก ยิ่งเข้มหรือใกล้ 1 ยิ่งดี
- ช่องนอกเส้นทแยง = ทายผิดเป็นคลาสอื่น
- แถวหรือคอลัมน์ `background` = วัตถุที่โมเดลมองไม่เห็น หรือทายว่ามีวัตถุในจุดที่ไม่มี

In [ ]:
# ===== Cell 10 : Confusion matrix =====
img = cv2.cvtColor(cv2.imread("runs/poly/confusion_matrix_normalized.png"), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(9, 9)); plt.imshow(img); plt.axis("off"); plt.show()

**อธิบาย Cell 11 : วัดผลกับชุด test** (ภาพที่โมเดลไม่เคยเห็นทั้งตอนเทรนและตอนเลือก weight)
- `.val(split="test")` = ให้โมเดลทายทุกภาพในชุด test แล้วเทียบกับ label จริง
- **Precision (P)** = ที่ทายว่าเจอ ถูกจริงกี่ % · **Recall (R)** = ของจริงทั้งหมด หาเจอกี่ %
- **mAP50** = ความแม่นเฉลี่ยทุกคลาส โดยนับว่าถูกเมื่อพื้นที่ซ้อนทับกับของจริง ≥ 50% และคลาสถูก
- **mAP50-95** = เข้มงวดกว่า เฉลี่ยตั้งแต่ซ้อนทับ 50% ถึง 95%
- มีทั้งผลของ **Box** (กรอบ) และ **Mask** (รูปร่าง)

In [ ]:
# ===== Cell 11 : วัดผลชุด test แยกรายคลาส =====
import pandas as pd
m = ultralytics.YOLO("weights/poly_best.pt").val(data="poly.yaml", split="test", imgsz=512, device="cpu",
                                                   project=RUNS, name="poly_test", exist_ok=True)
rows = [{"class": m.names[i], "box mAP50": round(m.box.ap50[i], 3), "mask mAP50": round(m.seg.ap50[i], 3), "box mAP50-95": round(m.box.ap[i], 3)}
        for i in range(len(m.names))]
rows.append({"class": "ALL", "box mAP50": round(m.box.map50, 3), "mask mAP50": round(m.seg.map50, 3), "box mAP50-95": round(m.box.map, 3)})
pd.DataFrame(rows)

---
## ขั้น 5 : Predict ด้วย weight ที่เทรนแล้ว

**อธิบาย Cell 12 :** โหลด `weights/poly_best.pt` แล้วทายภาพ test คลาสละ 1 ภาพ ใช้โค้ดวาดผลชุดเดียวกับขั้น 2
- `conf=0.25` = แสดงเฉพาะวัตถุที่โมเดลมั่นใจตั้งแต่ 25% ขึ้นไป
- ตอนนี้ชื่อคลาสเหลือแค่ 4 คลาสของเราแล้ว

In [ ]:
# ===== Cell 12 : Predict ภาพ test คลาสละ 1 ภาพ =====
model = ultralytics.YOLO("weights/poly_best.pt")
print("คลาสที่โมเดลรู้จักตอนนี้:", model.names)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, cls in zip(axes.flat, ["airplane", "boat", "car", "motorcycle"]):
    path = sorted(glob.glob(f'datasets/yolo_poly/test/images/{cls}_*.jpg'))[0]
    im_src = cv2.imread(path)
    result = model.predict(source=im_src, imgsz=512, conf=0.25, verbose=False)
    im_result = cv2.cvtColor(im_src, cv2.COLOR_BGR2RGB)

    if result[0].masks is not None:                       # ถ้าไม่เจอวัตถุเลย masks จะเป็น None
        for i in range(len(result[0].masks.xy)):          # วนทีละวัตถุ
            class_id = int(result[0].boxes.cls[i].item()) # เลขคลาส
            conf = round(result[0].boxes.conf[i].item()*100, 2)   # ความมั่นใจ %
            x = result[0].masks.xy[i].astype(int)         # จุดของ polygon [[x, y], ...] หน่วย pixel
            pts = x.reshape((-1, 1, 2))                   # จัดรูปแบบให้ตรงกับที่ cv2.polylines ต้องการ
            im_result = cv2.polylines(im_result, [pts], isClosed=True, color=(255, 0, 255), thickness=3)
            im_result = cv2.putText(im_result, f"{result[0].names[class_id]} {conf}%", (int(x[0][0]), max(int(x[0][1]) - 8, 20)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2, cv2.LINE_AA)

    ax.imshow(im_result); ax.set_title(f"true label: {cls}"); ax.axis("off")
plt.tight_layout(); plt.show()

**อธิบาย Cell 13 : ทายถูกกี่ภาพ?** ภาพแต่ละภาพมีคลาสเดียว ดูได้จากชื่อไฟล์ เช่น `car_013.jpg` คือคลาส car
- ให้โมเดลทายทุกภาพในชุด test แล้วเอาวัตถุที่ **มั่นใจที่สุด** เป็นคำตอบของภาพ
- เทียบกับชื่อไฟล์ แล้วนับว่าถูกกี่ภาพ แยกตามคลาส (ถ้าโมเดลไม่เจออะไรเลย จะนับเป็น "(ไม่เจอ)")

In [ ]:
# ===== Cell 13 : นับว่าทายถูกกี่ภาพ =====
rows = []
for f in sorted(glob.glob("datasets/yolo_poly/test/images/*.jpg")):
    r = model.predict(f, imgsz=512, conf=0.25, verbose=False)[0]
    true = os.path.basename(f).rsplit("_", 1)[0]                     # "car_013.jpg" -> "car"
    pred = r.names[int(r.boxes.cls[r.boxes.conf.argmax()])] if len(r.boxes) else "(ไม่เจอ)"
    rows.append({"true": true, "pred": pred, "correct": true == pred})
df = pd.DataFrame(rows)
print(f"ทายถูก {df.correct.sum()} / {len(df)} ภาพ  ({df.correct.mean()*100:.1f}%)")
pd.crosstab(df["true"], df["pred"], rownames=["จริง"], colnames=["ทาย"])

**อธิบาย Cell 14 :** เซฟผลการ predict ของทุกภาพใน test ไว้ส่งอาจารย์
- `save=True` ให้ ultralytics วาดผลลงภาพแล้วเซฟเอง ไว้ที่ `runs/predict_poly/`

In [ ]:
# ===== Cell 14 : เซฟผล predict ทั้งชุด test (ส่งอาจารย์) =====
model.predict(source="datasets/yolo_poly/test/images", imgsz=512, conf=0.25, save=True,
              project=RUNS, name="predict_poly", exist_ok=True, verbose=False)
saved = sorted(glob.glob("runs/predict_poly/*.jpg"))
print("runs/predict_poly/ :", len(saved), "ภาพ")

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, f in zip(axes, saved[::15]):
    ax.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); ax.set_title(os.path.basename(f)); ax.axis("off")
plt.tight_layout(); plt.show()

---
## ขั้น 6 : Export โมเดลไว้ใช้กับเว็บ

เว็บใช้โมเดล Polygon ได้ 2 แบบ จึงต้องเตรียมไฟล์ 2 ชนิด
| ใช้ที่ไหน | ไฟล์ | ใครเป็นคนรันโมเดล |
|---|---|---|
| หลังบ้าน FastAPI (`web/backend/app.py`) ตอนรันในเครื่อง | `weights/poly_best.pt` | Python + ultralytics บน server |
| หน้าเว็บออนไลน์ (Hugging Face, static) | `web/frontend/models/poly.onnx` | **เบราว์เซอร์** ของคนที่เปิดเว็บ (ไฟล์ `infer.js`) |

`weights/poly_best.pt` ถูก copy ไว้แล้วตอนจบ Cell 8 เหลือทำไฟล์ `.onnx`

**ONNX คืออะไร?** รูปแบบไฟล์โมเดลกลางที่หลายภาษาเปิดได้ ไม่ต้องมี Python หรือ PyTorch ใช้ **ONNX Runtime Web** (JavaScript) รันในเบราว์เซอร์ได้เลย hosting ฟรีที่รัน Python ไม่ได้ก็ยังใช้โมเดลได้

**อธิบาย Cell 15 :**
- `.export(format="onnx", imgsz=512)` แปลงโมเดลเป็น `.onnx` ขนาดภาพต้องเป็น **512 ให้ตรงกับ `IMGSZ` ใน `infer.js`**
- `simplify=True` ตัดส่วนที่ไม่จำเป็นออก ไฟล์เล็กลงและรันเร็วขึ้น
- ultralytics จะสร้างไฟล์ไว้ข้าง `.pt` แล้วเราย้ายไปไว้ที่ `web/frontend/models/poly.onnx`

In [ ]:
# ===== Cell 15 : Export เป็น ONNX ให้หน้าเว็บใช้ =====
os.makedirs("web/frontend/models", exist_ok=True)
onnx_path = ultralytics.YOLO("weights/poly_best.pt").export(format="onnx", imgsz=512, simplify=True)
shutil.move(onnx_path, "web/frontend/models/poly.onnx")

for f in ["weights/poly_best.pt", "web/frontend/models/poly.onnx"]:
    print(f"{f:<34} {os.path.getsize(f) / 1e6:.1f} MB")

**อธิบาย Cell 16 : ตรวจไฟล์ ONNX ว่าใช้ได้จริง**
- เปิดไฟล์ด้วย `onnxruntime` แล้วดู input/output
  - `images = [1, 3, 512, 512]` คือรับภาพ 1 ภาพ 3 สี (RGB) ขนาด 512×512
  - `output0 = [1, 40, 5376]` คือกล่องที่เป็นไปได้ 5376 กล่อง แต่ละกล่องมี 40 ค่า = `cx, cy, w, h` + คะแนน 4 คลาส + สัมประสิทธิ์ mask 32 ค่า
  - `output1 = [1, 32, 128, 128]` คือ mask พื้นฐาน 32 แผ่น เอามาผสมตามสัมประสิทธิ์จะได้ mask ของแต่ละวัตถุ
- `infer.js` ในเว็บอ่านค่าพวกนี้ แล้วกรองความมั่นใจ ทำ NMS (ตัดกรอบที่ซ้อนกัน) แปลงพิกัดกลับเป็นพิกัดของภาพจริง และสร้าง mask (อธิบายละเอียดใน `lab_final_web.ipynb`)
- สุดท้ายลองทายภาพเดียวกันด้วยไฟล์ `.pt` และ `.onnx` ผลต้องใกล้เคียงกัน อาจต่างกันเล็กน้อยเพราะ ONNX เติมขอบภาพให้เป็นสี่เหลี่ยมจัตุรัส 512×512 เสมอ

In [ ]:
# ===== Cell 16 : ตรวจไฟล์ ONNX =====
import onnxruntime as ort
sess = ort.InferenceSession("web/frontend/models/poly.onnx")
print("input :", [(i.name, i.shape) for i in sess.get_inputs()])
print("output:", [(o.name, o.shape) for o in sess.get_outputs()])

path = "datasets/yolo_poly/test/images/motorcycle_007.jpg"
for f in ["weights/poly_best.pt", "web/frontend/models/poly.onnx"]:
    r = ultralytics.YOLO(f, task="segment").predict(path, imgsz=512, conf=0.25, verbose=False)[0]
    print(f"{os.path.basename(f):<14} ->", [(r.names[int(c)], round(float(s), 3)) for c, s in zip(r.boxes.cls, r.boxes.conf)])

**อธิบาย Cell 17 : อัปโหลดเว็บขึ้น Hugging Face** (ทำหลัง export ทุกครั้งที่เทรนใหม่)
- สคริปต์ `scripts/05_deploy_huggingface.py` อัปโหลดทุกไฟล์ใน `web/frontend/` (หน้าเว็บและไฟล์ `poly.onnx`) ขึ้น Space แบบ static ที่ใช้ฟรี
- ตั้ง `DEPLOY = True` แล้วรัน cell (ต้อง login ไว้ก่อนด้วย `hf auth login`)
- เปิดเว็บได้ที่ https://witthaya22-final-dip.static.hf.space

In [ ]:
# ===== Cell 17 : Deploy เว็บ (ตั้ง DEPLOY = True เพื่ออัปโหลด) =====
import sys
DEPLOY = False
if DEPLOY:
    !{sys.executable} scripts/05_deploy_huggingface.py --space witthaya22/final-dip
else:
    print("ยังไม่ deploy (ตั้ง DEPLOY = True แล้วรันใหม่)")
print("เว็บ: https://witthaya22-final-dip.static.hf.space")